In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

# bits_and_bytes_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,  # Mixed precision
#     bnb_4bit_use_double_quant=True,       # Enable double quantization for efficiency
#     bnb_4bit_quant_type="nf4"             # Use NormalFloat4 for better accuracy
# )
# bits_and_bytes_config = BitsAndBytesConfig(
#     load_in_8bit=True,
# )

# Load pre-trained model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
cuda_enabled = torch.cuda.is_available()

In [2]:
# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # For language models
    r=16,                          # Rank of the low-rank matrices
    lora_alpha=16,                 # Scaling factor
    lora_dropout=0.1,              # Dropout for better regularization
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Move model to GPU
if cuda_enabled:
    device = torch.device("cuda")
    model.to(device)

In [ ]:
from elasticsearch import Elasticsearch
from dotenv import load_dotenv
import os

load_dotenv()

es = Elasticsearch(os.getenv("ES_HOST"), basic_auth=(os.getenv("ES_USER"), os.getenv("ES_PASSWORD")))

# Check connection
if es.ping():
    print("Connected to Elasticsearch")
else:
    print("Failed to connect to Elasticsearch")

response = es.search(
    index="transcripts",
    query={"match_all": {}},
    size=10000
)

documents = [
    {
        "instruction": "Generate an episode of The Amazing World Of Gumball with keywords: " + hit["_source"]["keywords"],
        "output": f"Title: {hit['_source']['title']}\n{hit['_source']['text']}"
    }
    for hit in response['hits']['hits']
]
print(f"Loaded {len(documents)} documents")

Connected to Elasticsearch
Loaded 256 documents


In [4]:
from datasets import Dataset

MAX_LENGTH=1024

splitted_documents = []
system_message = "You are a helpful AI assistant and you will help users generate episodes of The Amazing World Of Gumball."
for doc in documents:
    tokens = tokenizer(doc["output"], add_special_tokens=False)["input_ids"]
    old_part = None
    num_chunks = (len(tokens) + MAX_LENGTH // 2 - 1) // (MAX_LENGTH // 2)  # Calculate total number of chunks
    for i in range(0, len(tokens), MAX_LENGTH // 2):
        chunk = tokens[i:i + MAX_LENGTH // 2]
        new_part = tokenizer.decode(chunk, skip_special_tokens=True)
        is_last_chunk = i + MAX_LENGTH // 2 >= len(tokens)
        current_chunk = i // (MAX_LENGTH // 2) + 1  # Calculate current chunk number
        
        if old_part is None:
            chat = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": f"{doc['instruction']} (chunk {current_chunk}/{num_chunks})"},
                    {"role": "assistant", "content": new_part},
                ],
                tokenize=False
            )
        elif is_last_chunk:
            chat = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": system_message},
                    {"role": "assistant", "content": old_part},
                    {"role": "user", "content": f"Finish the episode... (chunk {current_chunk}/{num_chunks})"},
                    {"role": "assistant", "content": new_part},
                ],
                tokenize=False
            )
        elif old_part is not None:
            chat = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": system_message},
                    {"role": "assistant", "content": old_part},
                    {"role": "user", "content": f"Continue the episode... (chunk {current_chunk}/{num_chunks})"},
                    {"role": "assistant", "content": new_part},
                ],
                tokenize=False
            )
        old_part = new_part
        splitted_documents.append({"text": chat})

raw_dataset = Dataset.from_list(splitted_documents)
shuffled_dataset = raw_dataset.shuffle(seed=42)
tokenizer.pad_token = tokenizer.eos_token
train_test_split = shuffled_dataset.train_test_split(test_size=0.1)

def tokenize_function(example):
    tokenized = tokenizer(example["text"], padding='max_length', truncation=True, max_length=MAX_LENGTH+100)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = train_test_split.map(tokenize_function, batched=True)
print("Tokenized dataset")

# Move tokenized dataset to GPU
if cuda_enabled:
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    print("Moved tokenized dataset to GPU")

Map:   0%|          | 0/2103 [00:00<?, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

Tokenized dataset
Moved tokenized dataset to GPU


In [ ]:
# Train as usual
from transformers import Trainer, TrainingArguments, get_scheduler
from torch.optim import AdamW

model.train()

training_args = TrainingArguments(
    output_dir="./lora_results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    logging_steps=50,
    learning_rate=5e-04,
    weight_decay=0.01,
    use_cpu=False if cuda_enabled else True,
    lr_scheduler_type="cosine",
    fp16=True,
    warmup_ratio=0.1,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
)

class CustomTrainer(Trainer):
    def create_optimizer_and_scheduler(self, num_training_steps: int):
        self.optimizer = AdamW(
            self.model.parameters(),
            lr=self.args.learning_rate,
            betas=(0.9, 0.999),
            eps=1e-08,
            weight_decay=self.args.weight_decay,
        )
        self.lr_scheduler = get_scheduler(
            name=self.args.lr_scheduler_type,
            optimizer=self.optimizer,
            num_warmup_steps=int(num_training_steps * self.args.warmup_ratio),
            num_training_steps=num_training_steps,
        )

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

trainer.train()

Step,Training Loss,Validation Loss
100,10.162800,2.369528


In [ ]:
from datetime import datetime

# Save the model
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
model.save_pretrained(f"experiments/{current_time}")

In [ ]:
def generate_chat_response(keywords=None, max_new_tokens=1024, temperature=0.7, top_p=0.9, repetition_penalty=1.2, previous_part=None, chunk_number=None, num_chunks=None):
    if previous_part is None:
        chat = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": f"Generate an episode of The Amazing World Of Gumball with keywords: {keywords} (chunk {chunk_number}/{num_chunks})"},
            ],
            tokenize=False
        )
    else:
        chat = tokenizer.apply_chat_template(
            [
                {"role": "assistant", "content": previous_part},
                {"role": "user", "content": f"Continue the episode... (chunk {chunk_number}/{num_chunks})"},
            ],
            tokenize=False
        )
    encoded_chat = tokenizer.encode_plus(chat, return_tensors="pt")
    input_ids = encoded_chat["input_ids"].to(device)
    attention_mask = encoded_chat["attention_mask"].to(device)
    
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        use_cache=True
    )
    
    response = tokenizer.decode(output[0], skip_special_tokens=False)
    return response

In [ ]:
def generate_response(instruction, max_new_tokens=1024, temperature=0.7, top_p=0.9, repetition_penalty=1.2):
    inputs = tokenizer.encode_plus(
        instruction,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,  # Adjust temperature
        top_p=top_p,  # Use top-p sampling
        repetition_penalty=repetition_penalty,  # Apply repetition penalty
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    return response

In [ ]:
import re

def chat_template_split(chat):
    pattern = r"<\|im_start\|>(\w+)\s(.*?)<\|im_end\|>"
    matches = re.findall(pattern, chat, re.DOTALL)
    
    result = [{"role": role.lower(), "content": content.strip()} for role, content in matches]
    
    return result

In [ ]:
keywords = "gumball, darwin, school, trampoline, jumping a lot, race to school"
instruction = f"Keywords: {keywords}\n"
response = generate_chat_response(keywords, temperature=0.1, max_new_tokens=MAX_LENGTH//2, chunk_number=1, num_chunks=4)
print(response)
split_chat = chat_template_split(response)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Generate an episode of The Amazing World Of Gumball with keywords: gumball, darwin, school, trampoline, jumping a lot, race to school (chunk 1/4)<|im_end|>
<|im_start|>assistant
Title: The Jumping
Headline: School Day
[The episode starts in the Wattersons' living room. Gumball and Darwin are sitting on their bed]
Gumball: [Sighs] I guess it's time for us to get back into our normal lives. We've got work tomorrow morning!
Darwin: What about your homework?
Gumball: Oh, yeah... that one is pretty important. But we can't do anything else today because we're at school.
Darwin: Well, what if you could just jump around like this?!
Gumball: That would be awesome!
[They both start jumping up and down while laughing uncontrollably]
Gumball: Okay, so how much more does he have left before his next class?
Darwin: A little bit less than half-way there.
Gumball: So where should we go th

In [ ]:
print(split_chat)

[{'role': 'system', 'content': 'You are a helpful AI assistant named SmolLM, trained by Hugging Face'}, {'role': 'user', 'content': 'Generate an episode of The Amazing World Of Gumball with keywords: gumball, darwin, school, trampoline, jumping a lot, race to school (chunk 1/4)'}]


In [ ]:
next_response = generate_chat_response(previous_part=split_chat[-1]["content"], temperature=0.1, max_new_tokens=MAX_LENGTH//2, chunk_number=2, num_chunks=4)
print(next_response)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Generate an episode of The Amazing World Of Gumball with keywords: gumball, darwin, school, trampoline, jumping a lot, race to school (chunk 1/4)<|im_end|>
<|im_start|>user
Continue the episode... (chunk 2/4)<|im_end|>
<|im_start|>assistant
!
Headline: Jumping A Lot?
[Gumball and Darwin jump up]
Darwin: [Sighs] I guess it's just not that bad. We're still in third grade. It takes us forever to learn how to do this stuff. But we'll get there eventually.
Gumball: Yeah, but you know what they say about kids who don't have parents. They end up doing all kinds of stupid things like this.
Darwin: What kind of dumb thing is he talking about?!
Gumball: Well, for one, people think being overweight makes them look older than their age actually is. And then there's the fact that some people can be really rude when they see someone else wearing shorts or sneakers. So yeah, if your